# Panel Construction — Flood Exposure & Economic Activity

This notebook documents how the state-year processed CSVs in `data/processed/` are assembled
into the panel dataset used for regression (`final_regression_dataset.xlsx`).

Run from the repo root or any subdirectory — all paths use `REPO_ROOT` resolved from `__file__`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path().resolve()
# If running from notebooks/, move up one level
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

DATA = REPO_ROOT / 'data' / 'processed'
print('Repo root:', REPO_ROOT)
print('Data dir: ', DATA)

## 1. Load state-year CSVs

Each CSV is one row per Assembly Constituency (AC) for a given state and year.
State prefixes: `Bihar`, `Jharkhand`, `Odisha`, `WB` / `WestBengal`.

In [ ]:
STATE_PATTERNS = {
    'Bihar':      'Bihar*.csv',
    'Jharkhand':  'Jharkhand*.csv',
    'Odisha':     'Odisha*.csv',
    'WB':         'WB*.csv',
}
# WestBengalFloods_2015 is the renamed file (was WestBengalFloods_2015 _processed.csv)
WB_2015 = DATA / 'WestBengalFloods_2015_processed.csv'

frames = []
for state, pattern in STATE_PATTERNS.items():
    for f in sorted(DATA.glob(pattern)):
        df = pd.read_csv(f)
        df['_source_file'] = f.name
        frames.append(df)
        print(f'  {f.name}: {len(df):,} rows, {df.shape[1]} cols')

# West Bengal 2015 uses a different prefix
if WB_2015.exists():
    df = pd.read_csv(WB_2015)
    df['_source_file'] = WB_2015.name
    frames.append(df)
    print(f'  {WB_2015.name}: {len(df):,} rows, {df.shape[1]} cols')

raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal raw rows: {len(raw):,}')

## 2. Panel structure — shape per state

The panel should have ~765 unique ACs × 6 years (2014–2019) = 4,590 rows before merging.

In [ ]:
# Show shape per state — adjust column names if needed
if 'STATE' in raw.columns and 'YEAR' in raw.columns and 'AC_UID' in raw.columns:
    summary = (
        raw.groupby('STATE')
        .agg(
            rows=('AC_UID', 'count'),
            unique_ACs=('AC_UID', 'nunique'),
            years=('YEAR', 'nunique'),
            year_min=('YEAR', 'min'),
            year_max=('YEAR', 'max'),
        )
        .reset_index()
    )
    display(summary)
else:
    print('Column names in raw panel:', raw.columns.tolist())

## 3. Lag construction

Outcome variables require one-year lags. Year 2014 is kept in the panel only to serve as
the lagged baseline for 2015 — it is dropped from estimation.

Key lag columns created:
- `NL_median_t_minus_1`, `NL_mean_t_minus_1`
- `NDVI_median_t_minus_1`, `NDVI_mean_t_minus_1`
- `NDBI_median_t_minus_1`, `NDBI_mean_t_minus_1`

In [ ]:
# Load the final merged dataset and inspect lag columns
final = pd.read_excel(DATA / 'final_regression_dataset.xlsx')
print(f'Final dataset: {final.shape[0]:,} rows × {final.shape[1]} columns')

lag_cols = [c for c in final.columns if 't_minus_1' in c]
print(f'\nLag columns ({len(lag_cols)}):',  lag_cols)

# Estimation sample: drop 2014 (no t-1 covariates for NL)
est = final[final['YEAR'] > 2014].copy()
print(f'\nEstimation sample (2015-2019): {len(est):,} rows')

## 4. Summary statistics

In [ ]:
key_vars = [
    'Seasonal_Ratio', 'Permanent_Ratio',
    'NL_median', 'NL_mean',
    'NDVI_median', 'NDBI_median',
]
key_vars_present = [v for v in key_vars if v in est.columns]
display(est[key_vars_present].describe().round(4))

## 5. Correlation heatmap

In [ ]:
corr_vars = [v for v in key_vars_present if est[v].notna().sum() > 100]
corr = est[corr_vars].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Pairwise correlations — key variables (estimation sample 2015–2019)')
plt.tight_layout()
plt.show()